In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForMultipleChoice
import joblib

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

OPTS = ['A', 'B', 'C', 'D', 'E']
TEST_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Loading Test Data...")
test_df = pd.read_csv(TEST_PATH)
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    test_df[f'{col}_clean'] = test_df[col].apply(clean_text)

print(f"Test dataset shape: {test_df.shape}")


## MODEL 1 INFERENCE: FROM-SCRATCH PYTORCH OPTIONSCORER

In [ ]:
word_vec = joblib.load('models/scratch_word_vec.joblib')
char_vec = joblib.load('models/scratch_char_vec.joblib')
svd = joblib.load('models/scratch_svd.joblib')

def jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / len(sa | sb) if sa and sb else 0.0

def build_scratch_features(df):
    n = len(df)
    feats = np.zeros((n, 5, 8), dtype=np.float32)
    prompts = df['prompt'].fillna("").tolist()
    opt_texts = {c: df[c].fillna("").tolist() for c in OPTS}

    p_word = word_vec.transform(prompts)
    p_char = char_vec.transform(prompts)
    p_svd = svd.transform(p_word)

    opt_word = {c: word_vec.transform(opt_texts[c]) for c in OPTS}
    opt_char = {c: char_vec.transform(opt_texts[c]) for c in OPTS}
    opt_svd = {c: svd.transform(opt_word[c]) for c in OPTS}

    for i in range(n):
        p_len = max(len(prompts[i]), 1)
        p_words = max(len(prompts[i].split()), 1)
        word_cos = np.array([cosine_similarity(p_word[i], opt_word[c][i])[0, 0] for c in OPTS])
        char_cos = [cosine_similarity(p_char[i], opt_char[c][i])[0, 0] for c in OPTS]
        svd_cos = [cosine_similarity(p_svd[i:i+1], opt_svd[c][i:i+1])[0, 0] for c in OPTS]
        jac = [jaccard(prompts[i], opt_texts[c][i]) for c in OPTS]
        len_ratio = [len(opt_texts[c][i]) / p_len for c in OPTS]
        wc_ratio = [len(opt_texts[c][i].split()) / p_words for c in OPTS]

        opt_vecs = [opt_word[c][i] for c in OPTS]
        centrality = [np.mean([cosine_similarity(opt_vecs[a], opt_vecs[b])[0, 0] for b in range(5) if b != a]) for a in range(5)]
        word_cos_z = (word_cos - word_cos.mean()) / (word_cos.std() + 1e-6)

        for k in range(5):
            feats[i, k] = [word_cos[k], char_cos[k], svd_cos[k], jac[k], len_ratio[k], wc_ratio[k], centrality[k], word_cos_z[k]]
    return feats

X_test_raw = build_scratch_features(test_df)
num_feats = X_test_raw.shape[2]

In [ ]:
class OptionScorer(nn.Module):
    def __init__(self, in_features, hidden_dim=256, dropout=0.0):
        super(OptionScorer, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        B, K, F = x.shape
        return self.net(x.reshape(B * K, F)).reshape(B, K)

m2_test_probs_sum = np.zeros((len(test_df), 5))

In [ ]:
for fold in range(5):
    mean, std = joblib.load(f'models/scratch_scaler_fold{fold}.joblib')
    Xte_n = (X_test_raw - mean) / std
    Xte_t = torch.tensor(Xte_n, dtype=torch.float32).to(device)

    model = OptionScorer(num_feats).to(device)
    model.load_state_dict(torch.load(f'models/scratch_model_fold{fold}.pt', map_location=device))
    model.eval()

    with torch.no_grad():
        te_probs = torch.softmax(model(Xte_t), dim=1).cpu().numpy()
    m2_test_probs_sum += te_probs

m2_test_probs = m2_test_probs_sum / 5.0
print("Model 1 inference completed.")


## TRANSFORMER INFERENCE HELPER (DeBERTa & RoBERTa)

In [ ]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.prompts = df['prompt'].fillna("").tolist()
        self.options = df[OPTS].fillna("").values.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        prompt = self.prompts[idx]
        opts = self.options[idx]
        encoded = self.tokenizer(
            [prompt] * 5, opts,
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt'
        )
        return {k: v for k, v in encoded.items()}

In [ ]:
def predict_transformer(save_prefix):
    print(f"\n--- Inferencing Transformer ({save_prefix}) ---")
    tokenizer = AutoTokenizer.from_pretrained(f'models/{save_prefix}_tokenizer')
    test_dataset = MCQDataset(test_df, tokenizer, max_length=256)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

    test_probs_sum = np.zeros((len(test_df), 5))

    for fold in range(5):
        model = AutoModelForMultipleChoice.from_pretrained(f'models/{save_prefix}_fold{fold}').to(device)
        model.eval()

        fold_probs = []
        with torch.no_grad():
            for batch in test_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                    logits = model(**batch).logits
                fold_probs.append(torch.softmax(logits, dim=1).float().cpu().numpy())

        test_probs_sum += np.concatenate(fold_probs, axis=0)
        del model
        torch.cuda.empty_cache()

    return test_probs_sum / 5.0


In [ ]:
# Model 2: DeBERTa-v3-base Inference
m2_test_probs = predict_transformer("deberta_v3")

# Model 3: RoBERTa-base Inference
m3_test_probs = predict_transformer("roberta")